# **Understanding Word Space**

## **Set-Up**

In [151]:
# loading Python Libraries
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from collections import Counter
import pandas as pd
import seaborn as sns
import time

# Stopwords for text preprocessing
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.corpus import stopwords

# Sklearn Libraries for text processing and clustering
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Clustering and dimensionality reduction libraries
import umap
import hdbscan
import re


# Gensim FastText Model
from gensim.models import FastText

### **Load Data**

In [152]:
raw_data = pd.read_csv("data/UWSM_Community_Input_Transcripts_Split.csv")

print("--"*50)
print("Raw Data Shape:")
print(raw_data.shape)
print("--"*50)

----------------------------------------------------------------------------------------------------
Raw Data Shape:
(1769, 11)
----------------------------------------------------------------------------------------------------


In [153]:
print("--"*50)
print("Raw Data:")
print("--"*50)

raw_data.head()

----------------------------------------------------------------------------------------------------
Raw Data:
----------------------------------------------------------------------------------------------------


,Source File,Name of Facilitator(s),Date of Conversation,Name of Organization/Group,Meeting Location,Length of Time,Number of Attendees,Population,Summary Keywords,Speaker,Text
0,"EFO_AvestaOOB_York County, Older Adults, Rente...",Pamela Bennett,3/17/26,AVESTA residents,OOB,NaN,8,"Rural/urban, York/Cumberland/Mix, ALICE(Povert...","Affordable housing, food insecurity, community...",Speaker 1,"This is Pamela, and it's March 17, and I am at..."
1,"EFO_AvestaOOB_York County, Older Adults, Rente...",Pamela Bennett,3/17/26,AVESTA residents,OOB,NaN,8,"Rural/urban, York/Cumberland/Mix, ALICE(Povert...","Affordable housing, food insecurity, community...",Speaker 2,School System. The school system is specific t...
2,"EFO_AvestaOOB_York County, Older Adults, Rente...",Pamela Bennett,3/17/26,AVESTA residents,OOB,NaN,8,"Rural/urban, York/Cumberland/Mix, ALICE(Povert...","Affordable housing, food insecurity, community...",Speaker 3,I think the police department and fire departm...
3,"EFO_AvestaOOB_York County, Older Adults, Rente...",Pamela Bennett,3/17/26,AVESTA residents,OOB,NaN,8,"Rural/urban, York/Cumberland/Mix, ALICE(Povert...","Affordable housing, food insecurity, community...",Speaker 1,Any other strengths have to be specific to Old...
4,"EFO_AvestaOOB_York County, Older Adults, Rente...",Pamela Bennett,3/17/26,AVESTA residents,OOB,NaN,8,"Rural/urban, York/Cumberland/Mix, ALICE(Povert...","Affordable housing, food insecurity, community...",Speaker 1,friendship that we all have. We care for each ...


In [154]:
def text_length(text):
    return text.str.len()

In [155]:
raw_data["text_length"] = text_length(raw_data["Text"])

print("--"*50)
print("Transcript Length Statistics:")
print(raw_data["text_length"].describe())
print("--"*50)

----------------------------------------------------------------------------------------------------
Transcript Length Statistics:
count     1769.000000
mean       438.670435
std       1570.200479
min          6.000000
25%        135.000000
50%        264.000000
75%        500.000000
max      47036.000000
Name: text_length, dtype: float64
----------------------------------------------------------------------------------------------------


In [156]:
# Function to clean text data
def clean_text(text):

    # Remove special characters and numbers, keep only letters and spaces
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r'\b(Speaker)\s+(\d+)\b', r'\1\2', text)

    # Convert to lowercase
    text = text.lower()
    return text

In [157]:
# Apllying clean text function
raw_data["clean_text"] = raw_data["Text"].apply(clean_text)

print("--"*50)
print("Cleaned Data:")
print("--"*50)
raw_data.head()

----------------------------------------------------------------------------------------------------
Cleaned Data:
----------------------------------------------------------------------------------------------------


,Source File,Name of Facilitator(s),Date of Conversation,Name of Organization/Group,Meeting Location,Length of Time,Number of Attendees,Population,Summary Keywords,Speaker,Text,text_length,clean_text
0,"EFO_AvestaOOB_York County, Older Adults, Rente...",Pamela Bennett,3/17/26,AVESTA residents,OOB,NaN,8,"Rural/urban, York/Cumberland/Mix, ALICE(Povert...","Affordable housing, food insecurity, community...",Speaker 1,"This is Pamela, and it's March 17, and I am at...",384,this is pamela and its march 17 and i am at av...
1,"EFO_AvestaOOB_York County, Older Adults, Rente...",Pamela Bennett,3/17/26,AVESTA residents,OOB,NaN,8,"Rural/urban, York/Cumberland/Mix, ALICE(Povert...","Affordable housing, food insecurity, community...",Speaker 2,School System. The school system is specific t...,330,school system the school system is specific to...
2,"EFO_AvestaOOB_York County, Older Adults, Rente...",Pamela Bennett,3/17/26,AVESTA residents,OOB,NaN,8,"Rural/urban, York/Cumberland/Mix, ALICE(Povert...","Affordable housing, food insecurity, community...",Speaker 3,I think the police department and fire departm...,478,i think the police department and fire departm...
3,"EFO_AvestaOOB_York County, Older Adults, Rente...",Pamela Bennett,3/17/26,AVESTA residents,OOB,NaN,8,"Rural/urban, York/Cumberland/Mix, ALICE(Povert...","Affordable housing, food insecurity, community...",Speaker 1,Any other strengths have to be specific to Old...,106,any other strengths have to be specific to old...
4,"EFO_AvestaOOB_York County, Older Adults, Rente...",Pamela Bennett,3/17/26,AVESTA residents,OOB,NaN,8,"Rural/urban, York/Cumberland/Mix, ALICE(Povert...","Affordable housing, food insecurity, community...",Speaker 1,friendship that we all have. We care for each ...,271,friendship that we all have we care for each o...


### **Stop Words Removed**

In [158]:
# Stopwords for text preprocessing
nltk_stopwords = set(stopwords.words('english'))
sklearn_stopwords = set(ENGLISH_STOP_WORDS)

# Find unique stopwords in NLTK that are not in sklearn
nltk_unique = nltk_stopwords.difference(sklearn_stopwords)

# Combine sklearn and unique NLTK stopwords
stop_words_corpus = sklearn_stopwords.union(nltk_unique)

In [159]:
# Function to clean text data
def remove_stop_words(text):
    clean_text = []
    for word in text.split():
        if word not in stop_words_corpus:
            clean_text.append(word)
    
    return ' '.join(clean_text)

In [160]:
# Resmoving stopwords from cleaned conversations
raw_data["clean_text_no_stopwords"] = raw_data["clean_text"].apply(remove_stop_words)

In [161]:
# checking text length after cleaning
min_length = 1000000000
max_length = 0
for row, col in raw_data[["clean_text", "clean_text_no_stopwords"]].iterrows():
    print(f"Original: {len(col['clean_text'])} characters, No Stopwords: {len(col['clean_text_no_stopwords'])} characters")
    min_length = min(min_length, len(col['clean_text_no_stopwords']))
    max_length = max(max_length, len(col['clean_text_no_stopwords']))



Original: 372 characters, No Stopwords: 182 characters
Original: 313 characters, No Stopwords: 195 characters
Original: 455 characters, No Stopwords: 233 characters
Original: 104 characters, No Stopwords: 63 characters
Original: 258 characters, No Stopwords: 128 characters
Original: 105 characters, No Stopwords: 74 characters
Original: 201 characters, No Stopwords: 107 characters
Original: 162 characters, No Stopwords: 106 characters
Original: 238 characters, No Stopwords: 114 characters
Original: 76 characters, No Stopwords: 36 characters
Original: 471 characters, No Stopwords: 262 characters
Original: 124 characters, No Stopwords: 101 characters
Original: 239 characters, No Stopwords: 124 characters
Original: 119 characters, No Stopwords: 54 characters
Original: 119 characters, No Stopwords: 53 characters
Original: 64 characters, No Stopwords: 24 characters
Original: 557 characters, No Stopwords: 244 characters
Original: 127 characters, No Stopwords: 61 characters
Original: 207 chara

In [162]:
print("--"*50)
print("Minimum Text Length after Cleaning and Stopword Removal:")
print(min_length)
print("--"*50)
print("maximum Text Length after Cleaning and Stopword Removal:")
print(max_length)


----------------------------------------------------------------------------------------------------
Minimum Text Length after Cleaning and Stopword Removal:
0
----------------------------------------------------------------------------------------------------
maximum Text Length after Cleaning and Stopword Removal:
24518


## **Trainin Fastext-SkipGram**

In [163]:
complete_corpus = raw_data['clean_text_no_stopwords'].str.split().tolist()

# Find total number of tokens
total_tokens = sum(len(s) for s in complete_corpus)

# Find average number of tokens per document
avg_tokens = np.mean([len(s) for s in complete_corpus])

print("--"*50)
print("Preprocessing completed. Sample of cleaned text:")
print("--"*50)
print(f"Total number of tokens in the corpus: {total_tokens}")
print(f"Average number of tokens per document: {avg_tokens}")

----------------------------------------------------------------------------------------------------
Preprocessing completed. Sample of cleaned text:
----------------------------------------------------------------------------------------------------
Total number of tokens in the corpus: 59926
Average number of tokens per document: 33.875635952515545


In [164]:
print("--"*50)
print("Total number of documents in the corpus:", len(complete_corpus))
print("--"*50)
print("Sample of cleaned text (first 10 tokens of the first document):")
print("--"*50)
print(complete_corpus[0][:15]) 

----------------------------------------------------------------------------------------------------
Total number of documents in the corpus: 1769
----------------------------------------------------------------------------------------------------
Sample of cleaned text (first 10 tokens of the first document):
----------------------------------------------------------------------------------------------------
['pamela', 'march', '17', 'avesta', 'old', 'orchard', 'beach', 'reminder', 'im', 'going', 'ask', 'questions', 'answer', 'complete', 'moving']


In [165]:
start_time = time.time()

ftskipgram_model = FastText(
    sentences=complete_corpus,
    vector_size=300,      # embedding dimension
    window=5,             # context window
    min_count=5,          # ignore words that appear less than 5 times
    workers=4,            # parallelize training across 4 CPU cores
    epochs=25,            # number of training iterations
    sg=1,                 # Skip-gram
    min_n=3,              # Min length of char n-grams (default is 3)
    max_n=7,              # fewer subword combinations to learn
    negative=10,          # Negative samples. 10 is good for small data (default is 5)
    sample=1e-4,          # Downsample frequent words more aggressively (default 1e-3)
    alpha=0.025,          # Initial learning rate (default, but stating it)
    min_alpha=0.0001,     # Final learning rate
    seed=42,              # Reproducibility
)

time_ftskipgram = round(time.time() - start_time, 4)

print("--"*50)
print(f"Elapsed time for FastText Skip-gram: {time_ftskipgram:.4f} seconds")
print("Model Content:", ftskipgram_model)
print("--"*50)

----------------------------------------------------------------------------------------------------
Elapsed time for FastText Skip-gram: 5.2527 seconds
Model Content: FastText<vocab=1544, vector_size=300, alpha=0.025>
----------------------------------------------------------------------------------------------------


In [166]:
word_matrix = pd.DataFrame(
    ftskipgram_model.wv.vectors,
    index=ftskipgram_model.wv.index_to_key
)
print("--"*50)
print("First few rows of the FastText Skip-gram Word Embeddings Matrix:")
print("--"*50)
word_matrix

----------------------------------------------------------------------------------------------------
First few rows of the FastText Skip-gram Word Embeddings Matrix:
----------------------------------------------------------------------------------------------------


,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
like,0.017399,-0.043519,0.019649,0.028980,-0.086822,-0.046257,0.041911,-0.108795,-0.046292,0.024154,...,0.085612,-0.016485,-0.090448,-0.052984,0.002622,0.167182,-0.169944,-0.274521,0.094244,-0.092861
know,-0.055186,-0.020554,0.070513,0.159528,-0.019502,0.001796,0.033253,-0.045716,-0.070420,0.110048,...,0.041039,-0.000285,-0.094475,0.002748,-0.051654,0.219814,-0.229431,-0.202760,0.078540,-0.133089
think,0.027929,-0.089227,0.036075,0.105161,-0.062794,-0.025887,0.098752,-0.118384,-0.032391,0.102265,...,0.050763,0.031525,-0.083201,-0.030985,0.069261,0.098627,-0.125336,-0.208428,0.043262,-0.064184
people,-0.023715,-0.101777,0.059903,0.120797,-0.006185,-0.027186,0.041074,-0.128546,-0.046105,0.064906,...,0.023316,-0.005115,-0.114847,-0.053342,0.035132,0.160138,-0.206649,-0.256914,0.058425,-0.094573
yeah,-0.036857,-0.023134,0.027931,0.080902,-0.018255,0.026229,0.074272,-0.035348,-0.026663,0.111259,...,0.033982,0.012195,-0.108620,-0.027932,-0.060540,0.149427,-0.185475,-0.087069,0.102371,-0.132584
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
funny,-0.026688,-0.054513,0.041437,0.101640,-0.001582,-0.007466,0.056013,-0.054301,-0.039907,0.075284,...,0.035483,-0.013811,-0.143441,-0.022195,-0.018311,0.208724,-0.243593,-0.223525,0.112363,-0.130547
exact,0.016063,-0.058785,0.013472,0.096560,-0.042440,-0.012837,0.124342,-0.120680,-0.022488,0.120869,...,0.048274,0.046415,-0.094723,-0.048844,0.022718,0.024797,-0.027394,-0.040101,0.024158,-0.018240
summarize,-0.023335,-0.073817,0.030852,0.116165,-0.010168,0.002871,0.113420,-0.090365,-0.037895,0.113048,...,0.025917,0.021593,-0.101555,-0.027896,0.008970,0.115082,-0.137206,-0.132252,0.061136,-0.079373
watch,-0.064995,-0.024509,0.041207,0.112582,0.008899,0.021804,0.042752,-0.020264,-0.044967,0.070400,...,0.031705,-0.025897,-0.173908,-0.028136,-0.068491,0.272193,-0.310771,-0.215530,0.150961,-0.159473


### **Semantic Test**

In [167]:
# Define a list of labels to test semantic similarity
label_list = [
    "Housing Cost",
    "Cost of Basics",
    "Food Access",
    "Healthcare Cost",
    "Immigration Enforcement",
    "Behavioral Health",
    "Affordable Child Care Access",
    "Credit Card or Loan Payments",
    "Insurance Premiums",
    "Unexpected Expense of 400 Dollars",
    "Type of Employment",
    "Social Isolation"
]

# FastText Skip-gram Model Semantic Test
for label in label_list:
    print(f"Top 10 similar words to '{label}' in FastText Skip-gram model:")
    if label in ftskipgram_model.wv:
        similar_words = ftskipgram_model.wv.most_similar(label, topn=10)
        print(similar_words)
    else:
        print(f"  '{label}' not found in FastText Skip-gram vocabulary.")
    print("--"*50)

Top 10 similar words to 'Housing Cost' in FastText Skip-gram model:
[('struggling', 0.9713276028633118), ('increasing', 0.9667157530784607), ('advertising', 0.9652825593948364), ('housing', 0.9635167717933655), ('surrounding', 0.9621332883834839), ('continuing', 0.961571455001831), ('looking', 0.9589731693267822), ('impacting', 0.9502813220024109), ('networking', 0.9498628377914429), ('challenging', 0.948428213596344)]
----------------------------------------------------------------------------------------------------
Top 10 similar words to 'Cost of Basics' in FastText Skip-gram model:
[('perfect', 0.9900265336036682), ('silence', 0.9892301559448242), ('policy', 0.9888521432876587), ('picture', 0.9877146482467651), ('forward', 0.9874039888381958), ('separate', 0.9869375228881836), ('add', 0.9866340160369873), ('fine', 0.9851797819137573), ('directly', 0.984867513179779), ('springvale', 0.9848220348358154)]
-------------------------------------------------------------------------------

### **BERT-Topic Pipeline**

In [168]:
# Pulling Vocabulary and Embeddings from FastText Skip-gram Model
vocab = [w for w in ftskipgram_model.wv.key_to_index.keys()]
vectors = np.array([ftskipgram_model.wv[w] for w in vocab])

print(f"Vocab size: {len(vocab)}, vector shape: {vectors.shape}")

Vocab size: 1544, vector shape: (1544, 300)


In [169]:
TOP_N = 5000
word_freq = {w: ftskipgram_model.wv.get_vecattr(w, "count") for w in vocab}
top_words = sorted(word_freq, key=word_freq.get, reverse=True)[:TOP_N]
vocab = top_words
vectors = np.array([ftskipgram_model.wv[w] for w in vocab])

### **UMAP Reduction**

In [170]:
reducer_cluster = umap.UMAP(
    n_components=15,        
    n_neighbors=15,
    min_dist=0.0,           
    metric='cosine',        
    random_state=42,
)
emb_cluster = reducer_cluster.fit_transform(vectors)

reducer_viz = umap.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,          
    metric='cosine',
    random_state=42,
)
emb_viz = reducer_viz.fit_transform(vectors)

/Users/tejasphanse/miniconda3/envs/nlpenv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/tejasphanse/miniconda3/envs/nlpenv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### **HDBSCAN Clustering**

In [171]:
# min_cluster_size is the knob that matters most. Start at ~0.5% of vocab.
min_cluster_size = max(15, len(vocab) // 200)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=min_cluster_size,
    min_samples=5,
    metric='euclidean',     # UMAP output is already in euclidean space
    cluster_selection_method='eom',
)
labels = clusterer.fit_predict(emb_cluster)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()
print(f"Clusters: {n_clusters} | Noise points: {n_noise} ({n_noise/len(labels):.1%})")


Clusters: 2 | Noise points: 0 (0.0%)


In [172]:
# Constructing DataFrame for Visualization
df = pd.DataFrame({
    'word': vocab,
    'x': emb_viz[:, 0],
    'y': emb_viz[:, 1],
    'z': emb_viz[:, 2],
    'cluster': labels,
    'freq': [word_freq[w] for w in vocab],
})
print("--"*50)
print("DataFrame for Visualization:")
print("--"*50)
df

----------------------------------------------------------------------------------------------------
DataFrame for Visualization:
----------------------------------------------------------------------------------------------------


,word,x,y,z,cluster,freq
0,like,6.814487,6.550887,4.840132,1,5216
1,know,7.453096,7.061223,7.506884,1,1267
2,think,7.774588,3.176662,5.101305,1,1221
3,people,7.913298,4.530903,5.186235,1,1040
4,yeah,7.810619,3.268992,8.371456,1,1003
...,...,...,...,...,...,...
1539,funny,6.876555,6.553619,6.914252,1,5
1540,exact,8.399721,1.209073,7.182149,1,5
1541,summarize,8.093547,2.982545,7.092297,1,5
1542,watch,7.043933,7.825130,8.036565,1,5


In [173]:
# Drop noise for clarity, or keep it gray
df_plot = df[df['cluster'] != -1].copy()
df_plot['cluster_str'] = df_plot['cluster'].astype(str)

fig = px.scatter_3d(
    df_plot,
    x='x', y='y', z='z',
    color='cluster_str',
    hover_data=['word', 'freq'],
    opacity=0.7,
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(height=800, title=f"FastText word clusters ({n_clusters} clusters)")
fig.show()

### **Cluster labeling via corpus-level c-TF-IDF**

In [174]:
docs = raw_data['clean_text_no_stopwords'].tolist()
vectorizer = CountVectorizer(vocabulary=vocab, lowercase=False)
doc_term = vectorizer.fit_transform(docs)  # shape: n_docs x n_vocab
term_doc_freq = np.asarray((doc_term > 0).sum(axis=0)).flatten()  # df per word


In [175]:
cluster_summaries = []
for c in sorted(set(labels)):
    if c == -1:
        continue
    mask = df['cluster'] == c
    cluster_words = df.loc[mask, 'word'].tolist()
    cluster_freqs = df.loc[mask, 'freq'].tolist()

    # Distinctiveness: corpus frequency weighted down by how ubiquitous the word is
    word_indices = [vocab.index(w) for w in cluster_words]
    dfs = term_doc_freq[word_indices]
    scores = np.array(cluster_freqs) / np.log1p(dfs + 1)

    ranked = sorted(zip(cluster_words, scores), key=lambda x: -x[1])
    top_terms = [w for w, _ in ranked[:10]]

    cluster_summaries.append({
        'cluster': c,
        'size': int(mask.sum()),
        'top_terms': ', '.join(top_terms),
    })

summary_df = pd.DataFrame(cluster_summaries).sort_values('size', ascending=False)
print(summary_df.to_string(index=False))

 cluster  size                                                                          top_terms
       1  1406               like, know, think, people, yeah, community, thats, really, dont, lot
       0   138 going, thing, housing, talking, getting, thinking, coming, working, trying, saying


## **Weighted TF-IDF Construction**

In [176]:
# TF-IDF Vectorization of the cleaned conversations without stopwords
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(raw_data['clean_text_no_stopwords'])
feature_names = tfidf.get_feature_names_out()

In [177]:
# Function to compute weighted document vector using FastText embeddings and TF-IDF weights
def get_weighted_doc_vector(doc, model, tfidf_vec, row_idx):
    words = doc.split()
    # Map words to their TF-IDF feature index
    feature_index = {word: i for i, word in enumerate(feature_names)}
    
    vectors = []
    weights = []
    
    for word in words:
        if word in model.wv and word in feature_index:
            # Retrieve word vector and its statistical weight
            vec = model.wv[word]
            weight = tfidf_matrix[row_idx, feature_index[word]]
            
            vectors.append(vec * weight)
            weights.append(weight)
    
    # Average the weighted vectors
    return np.sum(vectors, axis=0) / np.sum(weights) if vectors else np.zeros(model.vector_size)

In [178]:
# Generate final vectors for all transcripts
final_transcript_vectors = [
    get_weighted_doc_vector(doc, ftskipgram_model, tfidf, i) 
    for i, doc in enumerate(raw_data['clean_text_no_stopwords'].tolist())
]

print(f"Generated {len(final_transcript_vectors)} weighted transcript vectors.")

Generated 1769 weighted transcript vectors.


In [179]:
# Printing the shape of the final transcript vectors
print(f"Shape of final transcript vectors: {np.array(final_transcript_vectors).shape}")

Shape of final transcript vectors: (1769, 300)


In [180]:
# Visualize the distribution of the final transcript vectors using UMAP
# Reduce dimensionality of the transcript vectors to 2D for visualization
umap_reducer = umap.UMAP(
    n_components=2,
    metric='cosine',
    random_state=42
)
transcript_vecs_2d = umap_reducer.fit_transform(np.array(final_transcript_vectors))

print(f"UMAP-reduced transcript vectors shape: {transcript_vecs_2d.shape}")

/Users/tejasphanse/miniconda3/envs/nlpenv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP-reduced transcript vectors shape: (1769, 2)


In [181]:
# Prepare hover text: show transcript index and first 15 words as a summary

df_umap = pd.DataFrame({
    "transcript_index": np.arange(len(complete_corpus)),
    "file_name": raw_data['Source File'].values,
    "x": transcript_vecs_2d[:, 0],
    "y": transcript_vecs_2d[:, 1],
    "summary": [
        (text[:15] + "..." if len(text.split()) > 15 else text)
        for text in raw_data['clean_text_no_stopwords']
    ]
})

In [182]:
print("--"*50)
print("UMAP Visualization DataFrame:")
print("--"*50)
df_umap

----------------------------------------------------------------------------------------------------
UMAP Visualization DataFrame:
----------------------------------------------------------------------------------------------------


,transcript_index,file_name,x,y,summary
0,0,"EFO_AvestaOOB_York County, Older Adults, Rente...",10.335331,2.798820,pamela march 17...
1,1,"EFO_AvestaOOB_York County, Older Adults, Rente...",5.686369,0.360615,school school s...
2,2,"EFO_AvestaOOB_York County, Older Adults, Rente...",2.800782,-0.752201,think police de...
3,3,"EFO_AvestaOOB_York County, Older Adults, Rente...",10.737014,3.093271,strengths specific old orchard broader communi...
4,4,"EFO_AvestaOOB_York County, Older Adults, Rente...",7.615643,0.722305,friendship care...
...,...,...,...,...,...
1764,1764,EFO_YLAT_3-11-26_Nina_Draft.txt,2.872999,-1.246096,like friendship reasons friendship race cats m...
1765,1765,EFO_YLAT_3-11-26_Nina_Draft.txt,6.812892,4.066787,im gonna read o...
1766,1766,EFO_YLAT_3-11-26_Nina_Draft.txt,9.867976,3.175162,fast know thank...
1767,1767,EFO_YLAT_3-11-26_Nina_Draft.txt,8.506544,1.938247,learn organizations pamphlets flyers today sti...


In [183]:
fig = px.scatter(
    df_umap,
    x="x",
    y="y",
    hover_data=["transcript_index", "summary"],
    title="UMAP Visualization of Transcript Vectors"
)
fig.update_traces(marker=dict(size=6, opacity=0.7))
fig.show()

In [184]:
(len(df_umap["summary"][2]))

18

In [185]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
 
 
def get_similar_words(model, label, topn=10):
    """
    Return [(word, score), ...] for a label that may be multi-word.
    Strategy:
      1. If the exact label key exists in wv, use it directly.
      2. Else, split into tokens; keep tokens present in wv; pass them as
         `positive=[...]` to most_similar (gensim averages them internally).
      3. If no tokens are in vocab, return None.
    """
    # Case 1: exact match (rare for multi-word labels)
    if label in model.wv:
        return model.wv.most_similar(label, topn=topn)
 
    # Case 2: token-level lookup
    tokens = label.lower().split()
    valid_tokens = [t for t in tokens if t in model.wv]
 
    if not valid_tokens:
        return None
 
    # Exclude the query tokens themselves from neighbors
    similar = model.wv.most_similar(positive=valid_tokens, topn=topn + len(valid_tokens))
    filtered = [(w, s) for w, s in similar if w not in valid_tokens][:topn]
    return filtered
 
 
def plot_label_similarities(model, label_list, topn=10, cols=3):
    """
    Create a grid of horizontal bar charts, one per label, showing the top-N
    most similar words and their cosine similarity scores.
    """
    rows = int(np.ceil(len(label_list) / cols))
 
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=label_list,
        horizontal_spacing=0.18,
        vertical_spacing=0.08,
    )
 
    # A single perceptually uniform colorscale so bars encode score twice
    # (length + color) — redundant but aids quick scanning across facets.
    colorscale = "Viridis"
 
    for idx, label in enumerate(label_list):
        row = idx // cols + 1
        col = idx % cols + 1
 
        results = get_similar_words(model, label, topn=topn)
 
        if results is None:
            # Draw a placeholder annotation inside the empty subplot
            fig.add_annotation(
                text="No tokens in vocab",
                xref=f"x{idx+1}" if idx > 0 else "x",
                yref=f"y{idx+1}" if idx > 0 else "y",
                x=0.5, y=0.5,
                showarrow=False,
                font=dict(size=11, color="gray"),
                row=row, col=col,
            )
            continue
 
        words, scores = zip(*results)
        # Reverse so highest score is at top when plotted horizontally
        words, scores = words[::-1], scores[::-1]
 
        fig.add_trace(
            go.Bar(
                x=scores,
                y=words,
                orientation="h",
                marker=dict(
                    color=scores,
                    colorscale=colorscale,
                    cmin=min(scores),
                    cmax=max(scores),
                    line=dict(width=0),
                ),
                text=[f"{s:.3f}" for s in scores],
                textposition="outside",
                hovertemplate="<b>%{y}</b><br>similarity: %{x:.4f}<extra></extra>",
                showlegend=False,
            ),
            row=row, col=col,
        )
 
        fig.update_xaxes(
            range=[0, max(scores) * 1.15],
            showgrid=True,
            gridcolor="rgba(0,0,0,0.06)",
            row=row, col=col,
        )
        fig.update_yaxes(
            automargin=True,
            row=row, col=col,
        )
 
    fig.update_layout(
        title=dict(
            text=f"Top {topn} Semantically Similar Words per Label (FastText Skip-gram)",
            x=0.5,
            font=dict(size=18),
        ),
        height=320 * rows,
        width=1200,
        template="plotly_white",
        margin=dict(l=60, r=40, t=90, b=40),
    )
 
    # Shrink facet titles so they don't crowd
    for ann in fig.layout.annotations:
        if ann.text in label_list:
            ann.font = dict(size=12, color="#222")
 
    return fig

In [186]:
    fig = plot_label_similarities(ftskipgram_model, label_list, topn=10, cols=3)
    fig.show()